## 0 - Imports


In [1]:
#Basic
import pandas as pd
import numpy as np
import json
import math

#Natural Language
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import re



import string

#Indexing/Scoring
from collections import defaultdict
from rank_bm25 import BM25Okapi

#Embeddings
from sentence_transformers import SentenceTransformer

#Machine Learning
from sklearn.metrics.pairwise import cosine_similarity


## 1 - Data Extraction
- Load JSONs
- See what is available

Use this link https://business.yelp.com/data/resources/open-dataset/
and download the JSON .zip - once downloaded extract it into your repo with the folder within the .gitignore
and then you can run the following code!

In [3]:
#Extracting data - was going to do Phoenix however it is not in the data so we will do all of Arizona

business = pd.read_json('../data/yelp_academic_dataset_business.json', lines=True)
# filtered by Arizona
business = business[business['state'] == 'AZ']
# filter by food/restuarant string
business = business[business['categories'].str.contains('Restaurant | Food',case = False,na=False)
                    &
                    ~business['categories'].str.contains('Drugstore|Convenience|Automotive|Grocery|Store',case = False, na=False)]
#Filter by open
business = business[business['is_open']==1]
#Filter by review count greater than 5
business = business[business['review_count']>5]
business



,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
56,txyXRytGjwOXvS8s4sc-WA,Smoothie King,1070 E Tucson Marketplace Blvd,Tucson,AZ,85713,32.186794,-110.954765,3.0,29,1,"{'RestaurantsPriceRange2': '2', 'BusinessParki...","Vitamins & Supplements, Ice Cream & Frozen Yog...","{'Monday': '0:0-0:0', 'Tuesday': '7:0-21:0', '..."
319,f82dhKNiUXsDVPMLqKYiIQ,Sher-e-Punjab,853 East Grant Rd,Tucson,AZ,85719,32.250960,-110.959158,4.0,446,1,"{'RestaurantsAttire': ''casual'', 'BusinessAcc...","Restaurants, Salad, Pakistani, Indian, Cocktai...","{'Tuesday': '16:0-21:0', 'Wednesday': '16:0-21..."
553,adATTqggIQX5xxLDISkFTw,Just Churros,,Tucson,AZ,85705,32.271231,-110.992075,5.0,25,1,"{'BusinessAcceptsCreditCards': 'True', 'Restau...","Food Trucks, Restaurants, Caterers, Event Plan...","{'Monday': '0:0-0:0', 'Friday': '15:0-21:0', '..."
954,2vAqYNN86VWXZiy2E96-TQ,Chick-fil-A,"1303 E University Blvd, Ste 149",Tucson,AZ,85719,32.232445,-110.951699,3.0,13,1,"{'RestaurantsReservations': 'False', 'GoodForK...","Event Planning & Services, Caterers, Fast Food...","{'Monday': '0:0-0:0', 'Tuesday': '10:0-17:0', ..."
995,iNMdSi5bmvGSGeRQiUW4dw,Wendy's,3535 E. Irvington Road,Tucson,AZ,85714,32.163740,-110.916722,2.5,14,1,"{'BusinessAcceptsCreditCards': 'True', 'Restau...","Fast Food, Burgers, Restaurants","{'Monday': '10:0-23:0', 'Tuesday': '10:0-23:0'..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149247,zfw03c1jP7sYkIfu1da64w,Panda Express,"9565 E. 22nd Street, SUITE 155",Tucson,AZ,85748,32.206846,-110.788470,2.0,45,1,"{'Ambience': '{'romantic': False, 'intimate': ...","Restaurants, Fast Food, Chinese","{'Monday': '10:30-21:30', 'Tuesday': '10:30-21..."
149436,NHz8uMabvQ2nXk6CddCK4w,McDonald's,3315 N Swan,Tucson,AZ,85712,32.267066,-110.893169,2.5,20,1,"{'RestaurantsAttire': ''casual'', 'Restaurants...","Fast Food, Burgers, Restaurants, Coffee & Tea,...","{'Monday': '5:0-23:0', 'Tuesday': '5:0-23:0', ..."
150000,aGOXuqO6yhN66tLYI61Thg,Jack in the Box,4450 1st Ave,Tucson,AZ,85719,32.287556,-110.960460,4.5,13,1,"{'DriveThru': 'True', 'Caters': 'False', 'Bike...","Tacos, American (Traditional), Fast Food, Mexi...","{'Monday': '0:0-0:0', 'Tuesday': '0:0-0:0', 'W..."
150127,K_kRU8j8th6yBeLbI94pJQ,Starbucks,555 E Grant Rd,Tucson,AZ,85705,32.251672,-110.962574,3.5,9,1,"{'RestaurantsTakeOut': 'True', 'BusinessParkin...","Coffee & Tea, Food","{'Monday': '6:0-19:0', 'Tuesday': '6:0-19:0', ..."


In [4]:
#Gather IDs
ids = business['business_id'].to_list()
# Append reviews safely
reviews = []
with open('../data/yelp_academic_dataset_review.json','r', encoding = 'utf-8') as f:
    for line in f:
        review = json.loads(line)
        if review['business_id'] in ids:
            reviews.append(review)

rev_df = pd.DataFrame(reviews)
rev_df

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,onlgwy5qGDEzddsrnIvtWg,pYXeL0RCqus2IfhthYCOyA,W7NxQw8UYFR0HLPrI08tvw,4.0,0,0,0,Don't know what it is but If my tummy's feelin...,2012-02-01 14:21:25
1,mRnYZes0nj4sr8DsE_gWMQ,FuTJWFYm4UKqewaosss1KA,fgTOJRkc703E4XRdcr5zRA,3.0,3,0,0,I've come from Cali where boba is very common ...,2016-01-30 01:59:11
2,f7fAYGJpd4gZAoJxuJcciw,LpZfJekvMo5S61UBAmuyHw,cXAKeC-EgVChIxhS7fscmw,5.0,2,0,0,The food at Ghini's is just delicious. Everyth...,2011-05-27 15:44:29
3,b0AI6U9CCWpFKI3iPLYiow,_l0csyXqNIcb3vG-1qR8DQ,UCMSWPqzXjd7QHq7v8PJjQ,4.0,0,0,0,I really like Prep & Pastry - we have been twi...,2017-09-20 18:19:33
4,Zssrl36KBW-QMHsa8G9a_w,zcYZgNeJHpKCSBRwh6WskQ,SbdL-8NSmTWgSwdGZBa7WQ,5.0,0,0,0,Another five-star dining review! Fresco serve...,2015-02-24 04:27:38
...,...,...,...,...,...,...,...,...,...
78857,VkXr54yJMN4Qu4dRznAN8Q,4jEdEPDNAAa3aS7rYhQ60w,MK0OMY_u9unl8xSqjPLtMw,5.0,3,0,0,"ALWAYS love this place! And it's always busy, ...",2020-01-17 20:58:05
78858,JA8GCU3glb6TQRExleWjHg,Iq-9jCp219AEcbtjy-ZyNQ,tBWjMqUc0yP5lRElCfDaKg,5.0,1,0,1,"Great pizza, awesome choices of beers, pet fri...",2016-07-25 01:08:01
78859,PMgEv05rnLIJZlpGc4IHvQ,CDUkT2tD6y3gSfivf3Beyw,EhFJgjgn9Kzo_gu03DVkRg,3.0,1,0,0,Not my favorite dessert place. You don't get m...,2018-02-05 17:27:07
78860,jacDcaIWSPdZq2bDq1GD_g,Y-mwrjOx29pnJX0MCBb2Yg,9VRmMY9vGhGKGz9hiGoEUw,1.0,0,0,0,If I could leave no stars I would. I understan...,2021-11-28 14:23:39


In [5]:
#Concatenate maximum 20 reviews for each unique business_id
group_rev = (rev_df.groupby('business_id')['text'].apply(lambda x: " ".join(x[:20])).reset_index())
group_rev

,business_id,text
0,-1w9JMktu9oWTXwNqtZQoA,I was in Tuscan from Baltimore md . I stumbled...
1,-3-6BB10tIWNKGEF0Es2BA,We will absolutely be coming back here! The ch...
2,-7cNgs6N105MDlLjOudObg,I'm a regular here for sure. Fresh ingredients...
3,-Ah16__ceG91aXtrbkhkxQ,Yummy cakes! Always fresh and they taste great...
4,-B6fyJ8PoAMr_mH5VGaPjA,"Best Bacon wrapped burritos in Town, come in e..."
...,...,...
969,zbhID412Pg3zXd_t3mswpg,Finally! A last minute BB fix before you leave...
970,zfw03c1jP7sYkIfu1da64w,First time eating at this location. The inside...
971,zgClnCzcLl1gzUeKFaJAhg,The food was good the service was slow but lat...
972,zkrEIgrkGylMek2-dUZgZg,I tried Popeyes again 3 months after the 1st r...


In [6]:
# Merge business data to get documents
docs = business.merge(group_rev, on='business_id')

docs['document'] = (
    'Name: ' + docs['name'] + '\n' +
    'Categories: ' + docs['categories'] + '\n' +
    'Ratings: ' + docs['stars'].astype(str) + '\n' +
    'Reviews: ' + docs['text'].fillna('')

)
docs['document']

0      Name: Smoothie King\nCategories: Vitamins & Su...
1      Name: Sher-e-Punjab\nCategories: Restaurants, ...
2      Name: Just Churros\nCategories: Food Trucks, R...
3      Name: Chick-fil-A\nCategories: Event Planning ...
4      Name: Wendy's\nCategories: Fast Food, Burgers,...
                             ...                        
969    Name: Panda Express\nCategories: Restaurants, ...
970    Name: McDonald's\nCategories: Fast Food, Burge...
971    Name: Jack in the Box\nCategories: Tacos, Amer...
972    Name: Starbucks\nCategories: Coffee & Tea, Foo...
973    Name: Savaya Coffee Market\nCategories: Specia...
Name: document, Length: 974, dtype: object

In [7]:
# Save corpus of documents (restaurant data) as .pkl - no need to run Step 1: again after this
# From now on use .pkl
docs['document'].to_pickle('../data/arizona_restuarant_corpus.pkl')

## 2 - Preprocessing
- lowercase
- tokenization
- stemming (removed for now)
- stopword removal


In [2]:
documents = pd.read_pickle('../data/arizona_restuarant_corpus.pkl')
documents

0      Name: Smoothie King\nCategories: Vitamins & Su...
1      Name: Sher-e-Punjab\nCategories: Restaurants, ...
2      Name: Just Churros\nCategories: Food Trucks, R...
3      Name: Chick-fil-A\nCategories: Event Planning ...
4      Name: Wendy's\nCategories: Fast Food, Burgers,...
                             ...                        
969    Name: Panda Express\nCategories: Restaurants, ...
970    Name: McDonald's\nCategories: Fast Food, Burge...
971    Name: Jack in the Box\nCategories: Tacos, Amer...
972    Name: Starbucks\nCategories: Coffee & Tea, Foo...
973    Name: Savaya Coffee Market\nCategories: Specia...
Name: document, Length: 974, dtype: object

In [ ]:
#Preprocess - appears to work better for vocab list
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    tokens = text.lower().split() #lower case all
    tokens = [x for x in tokens if x not in stop_words and x not in string.punctuation] #no stop words or punctuations
    # tokens = [stemmer.stem(x) for x in tokens] #stemming
    return tokens

In [ ]:
#Preprocess V2 - appears to work better for cleaning up queries to be meaningful
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocessv2(text):
    #lowercase text
    text = text.lower()

    # tokenize words
    tokens = re.findall(r"\b[a-z]+\b",text)

    #tokenization
    # tokens = text.split()

    #removing stop words
    tokens = [x for x in tokens if x not in stop_words]

    # lemmatize 
    tokens = [lemmatizer.lemmatize(x) for x in tokens]
    return tokens

In [120]:
# Apply to documents
tokens0 = [preprocess(doc) for doc in documents]
tokens = [preprocessv2(doc) for doc in documents]

print(f'Original: {tokens0[0][:5]} --> New: {tokens0[0][:5]}')


Original: ['name:', 'smoothie', 'king', 'categories:', 'vitamins'] --> New: ['name:', 'smoothie', 'king', 'categories:', 'vitamins']


## 3 - Indexing | Scoring (Initial Test)
- Create the Inverted Index 
- TF-IDF testing
- BM25 testing
- GOAL: Make the inverted index and just check and see how scoring works 

In [121]:
# INVERTED INDEX

invert_index = defaultdict(dict)

for id, token in enumerate(tokens):
    for x in token:
        invert_index[x][id] = invert_index[x].get(id,0)+1

#example 

print(invert_index['taco'])



{8: 3, 9: 26, 12: 1, 24: 3, 32: 26, 35: 4, 36: 18, 38: 1, 40: 1, 41: 1, 43: 19, 73: 2, 76: 2, 79: 1, 83: 26, 90: 19, 96: 19, 99: 1, 105: 4, 106: 4, 110: 2, 111: 19, 112: 6, 115: 1, 116: 1, 119: 3, 120: 4, 126: 18, 127: 5, 131: 3, 141: 6, 146: 4, 151: 6, 154: 8, 158: 1, 172: 7, 179: 25, 180: 4, 181: 27, 187: 4, 189: 14, 190: 2, 191: 2, 201: 1, 218: 1, 219: 2, 224: 2, 228: 2, 229: 2, 231: 1, 234: 1, 243: 3, 248: 1, 251: 1, 263: 4, 264: 23, 266: 1, 270: 13, 272: 2, 278: 13, 284: 22, 285: 4, 287: 5, 300: 6, 305: 19, 312: 26, 320: 1, 323: 4, 326: 9, 330: 1, 345: 1, 353: 15, 361: 22, 364: 7, 365: 2, 375: 5, 383: 10, 390: 21, 392: 13, 393: 27, 402: 1, 403: 2, 407: 8, 409: 2, 421: 4, 426: 1, 430: 3, 432: 1, 435: 2, 436: 1, 445: 1, 447: 20, 451: 7, 455: 2, 460: 10, 464: 1, 472: 32, 474: 16, 475: 5, 492: 21, 495: 1, 501: 1, 509: 12, 521: 43, 522: 1, 526: 1, 530: 2, 534: 31, 535: 1, 549: 16, 552: 1, 554: 1, 556: 2, 561: 8, 567: 15, 570: 6, 571: 3, 579: 2, 591: 2, 592: 1, 595: 2, 597: 7, 628: 10, 

In [122]:
# Basic TF-IDF

# N - total # docs
# df - number of docs with term
# idf - log(N/df)


N = len(tokens)

def tfidf(term):
    df = len(invert_index[term])
    return math.log(N/df)


tfidf_score = tfidf('taco')

print(tfidf_score)


#BUILD IDF DICTIONARY
idf = {}
for term in invert_index:
    df = len(invert_index[term])
    idf[term] = math.log(N/df)

#maybe add l2 normalization to make it 0-1 scale

1.5584013245041268


In [126]:
# BM25 Implementation
bm25 = BM25Okapi(tokens)

query = preprocessv2('best tacos')

scores = bm25.get_scores(query)

print('Before Ranking')
print(scores[:5])
ranked_docs = np.argsort(scores)[::-1]
print(f'\nAfter Ranking\n{ranked_docs[:5]}')

Before Ranking
[2.39108109 2.66423091 2.63177683 1.54282944 1.65668543]

After Ranking
[472 715  36 859 709]


In [127]:
#TF-IDF scoring logic
N = len(tokens)

def tfidf_scoring(query):
    scores = [0] * N

    for x in query:
        if x in invert_index:
            for id,tf in invert_index[x].items():
                # scores[id]+=tf*idf[x] #basic
                scores[id] += (tf / len(tokens[id])) *idf[term] #length normalization - way more similar to BM25

    return scores

In [128]:
#Compare Scores
#TF-IDF
tf_scores = tfidf_scoring(query)

#BM25
bm25_scores = bm25.get_scores(query)

def top_results(scores,x=5):
    return np.argsort(scores)[::-1][:x] #top 5 results

tf_top = top_results(tf_scores)
bm_top = top_results(bm25_scores)

print(f'TF-IDF Top Results: ')
for rank,id in enumerate(tf_top,1):
    print(f'{rank} -> (Document: {id}) -> Score: {tf_scores[id]:.2f}')
    print('  ', tokens[id])

print('\nBM25 Top Results: ')
for rank, id in enumerate(bm_top,1):
    print(f'{rank} -> (Document: {id}) -> Score: {bm25_scores[id]:.2f}')
    print('   ', tokens[id])

TF-IDF Top Results: 
1 -> (Document: 709) -> Score: 0.48
   ['name', 'taco', 'rico', 'category', 'food', 'truck', 'food', 'taco', 'restaurant', 'fast', 'food', 'mexican', 'rating', 'review', 'absolutely', 'delicious', 'best', 'mexican', 'truck', 'tucson', 'glad', 'located', 'ina', 'driving', 'south', 'sometimes', 'drag', 'waste', 'time', 'writing', 'review', 'going', 'show', 'day', 'write', 'many', 'review', 'deleted', 'well', 'sure', 'owner', 'business', 'since', 'gave', 'star', 'review', 'assume', 'yelp', 'great', 'taco', 'taco', 'truck', 'specializes', 'taco', 'ordered', 'chicken', 'carne', 'asada', 'steak', 'taco', 'came', 'sliced', 'radish', 'grilled', 'onion', 'cilantro', 'sauce', 'tried', 'pastor', 'asada', 'cabeza', 'taco', 'delicious', 'tortilla', 'excellent', 'meat', 'fresh', 'moist', 'served', 'radish', 'cucumber', 'onion', 'side', 'plus', 'traveled', 'oro', 'valley', 'say', 'worth', 'trip', 'back', 'soon', 'food', 'truck', 'actually', 'really', 'good', 'tried', 'asada', 'ca

## 4 - Query Expansion
- Expand the query using embeddings
- Also Testing using Wordnet for synonyms since my dataset is small
- Goal: Create a function with both and test results later

In [ ]:
# use sentence transformer model
model_s = SentenceTransformer('all-MiniLM-L6-v2')
# Retrieve index tokens and embedded the vocabulary
vocab = list(invert_index.keys())
embeddings = model_s.encode(vocab)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [150]:
#Query expansion func
def query_expand(query):
    #tokenize query
    # q_tokens = query.lower().split()
    q_tokens = preprocessv2(query)
    # print(q_tokens)

    #embedding for query
    query_embeddings = model_s.encode(query)

    #dot product vocab * query embeddings
    prod = cosine_similarity([query_embeddings],embeddings)[0]
    top = np.argsort(prod)[-3:][::-1]
    # expansion.extend([vocab[i] for i in top if vocab[i] not in q_tokens])
    expansion = [vocab[i] for i in top if vocab[i] not in q_tokens]
    return q_tokens + expansion


In [151]:
# Testing query expansion via embedding OUTPUT -> new query
query_expand('What are the best vegan taco shops?')

['best', 'vegan', 'taco', 'shop', 'taquitos', 'tacobell']

Now lets test wordnet for synonym based query expansion


In [95]:
#Wordnet Synonym expansion
def syn_expand(query):
    # query = query.lower().split() #basic tokenization
    q_tokens = preprocessv2(query) #consistent approach
    # print(q_tokens)
    #Set - for unique values
    expanded = set(q_tokens)

    for word in q_tokens:
        syn = wordnet.synsets(word)

        for syns in syn[:2]: #2 synonym limit
            for lemma in syns.lemmas():
                expanded.add(lemma.name().replace("_", " "))
    
    return list(expanded)


In [136]:
syn_expand('What are the best vegan shops?') #taco was bad

['best', 'workshop', 'shop', 'vegan', 'topper', 'store']

Okay lets not use synonym based expansion that was unexpected

## 5 - Semantic Embedding Similarity


In [65]:
doc_embeds = model_s.encode(documents, show_progress_bar=True)

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

In [186]:
#Semantic Search Func
def semantic_search(query):

    # preprocess query using preprocessv2
    q_tokens = preprocessv2(query)

    #Query Expansion
    exp_tokens = query_expand(query)

    extra_tokens = [x for x in exp_tokens if x not in q_tokens] # just new query words that were added in the expansion of the query
    exp_text = " ".join(q_tokens + extra_tokens)

    
    # vector = model_s.encode(query)
    vector = model_s.encode(exp_text)


    similarity = cosine_similarity([vector],doc_embeds)[0]

    top_doc = np.argsort(similarity)[-10:][::-1] #top 10 ordered descending

    for x in range(10):
        doc_id = top_doc[x]
        score = similarity[doc_id]
        print(f'Score: {score:.2f} --> {tokens[doc_id]}')
    return top_doc

In [187]:
semantic_search('What are the best vegan taco shops?')

Score: 0.67 --> ['name', 'la', 'chaiteria', 'category', 'mexican', 'taco', 'vegan', 'vegetarian', 'food', 'truck', 'food', 'restaurant', 'rating', 'review', 'hungry', 'friday', 'afternoon', 'lunch', 'looking', 'restaurant', 'driving', 'saw', 'new', 'place', 'dropped', 'ordered', 'al', 'pastor', 'taco', 'raja', 'taco', 'green', 'chile', 'mushrooom', 'cream', 'al', 'pastor', 'taste', 'authentic', 'oily', 'raja', 'tasteless', 'bland', 'horrible', 'wow', 'either', 'another', 'note', 'place', 'clean', 'tidy', 'stopped', 'today', 'pick', 'large', 'order', 'family', 'taco', 'night', 'everything', 'tasted', 'incredible', 'fresh', 'ordered', 'taco', 'al', 'pastor', 'mole', 'taco', 'definitely', 'back', 'happy', 'wendy', 'opened', 'cafe', 'closer', 'neck', 'wood', 'great', 'menu', 'item', 'featuring', 'best', 'vegetarian', 'food', 'cafe', 'vegetarian', 'love', 'magic', 'food', 'wendy', 'creates', 'potato', 'filled', 'taco', 'jackfruit', 'carnitas', 'go', 'wrong', 'happen', 'day', 'cashew', 'base

array([751, 549, 361, 715,  43, 730, 709, 647, 318, 842], dtype=int64)

## 6 - Hybrid Scoring Approach
- Combine both BM25 Results with Semantic Searching with a split % between their scores
- GOAL: To achieve a moderate output based off the strenghts of both methods

In [228]:
def hybrid_search(query,a =.2):

    # preprocess query using preprocessv2
    q_tokens = preprocessv2(query)

    #Query Expansion
    exp_tokens = query_expand(query)

    extra_tokens = [x for x in exp_tokens if x not in q_tokens] # just new query words that were added in the expansion of the query
    exp_text = " ".join(q_tokens + extra_tokens)

    # print(extra_tokens)
    print(f'Query: {exp_text}')

    #BM25 Scoring - weight original query strongest followed by little of expansion
    bm25_scoring = (
        1*np.array(bm25.get_scores(q_tokens)) +  .3*np.array(bm25.get_scores(extra_tokens))
    )

    #Semantic Scoring
    vector = model_s.encode(exp_text)
    semantic_scoring = cosine_similarity([vector],doc_embeds)[0]

    #Combination
    final = a*np.array(bm25_scoring) + (1-a) * semantic_scoring

    top_documents = np.argsort(final)[-10:][::-1]
    for x in range(10):
        doc_id = top_documents[x]
        score = final[doc_id]
        print(f'Score: {score:.2f} --> {tokens[doc_id]}')

    return top_documents
    

In [229]:
hybrid_search('What are the best vegan taco shops?')

Query: best vegan taco shop taquitos tacobell
Score: 2.66 --> ['name', 'taco', 'stop', 'category', 'caterer', 'event', 'planning', 'service', 'food', 'truck', 'food', 'restaurant', 'mexican', 'rating', 'review', 'excellent', 'taco', 'price', 'reasonable', 'quality', 'quantity', 'received', 'hope', 'around', 'next', 'visit', 'tuscon', 'wanted', 'know', 'hype', 'tried', 'jackfruit', 'burro', 'mind', 'vegan', 'vegetarian', 'wanted', 'try', 'also', 'actually', 'hate', 'jackfruit', 'asian', 'family', 'think', 'love', 'big', 'fan', 'kind', 'fruit', 'durian', 'jackfruit', 'lychee', 'etc', 'think', 'eating', 'jackfruit', 'burro', 'long', 'long', 'time', 'marinaded', 'something', 'good', 'felt', 'like', 'best', 'burro', 'far', 'life', 'soooo', 'big', 'usually', 'never', 'finish', 'one', 'whole', 'burro', 'size', 'one', 'sitting', 'ate', 'put', 'good', 'mood', 'texted', 'sister', 'florida', 'good', 'service', 'great', 'super', 'friendly', 'food', 'fresh', 'made', 'order', 'glad', 'moved', 'would

array([361, 751, 820, 263, 284, 549, 883, 529, 318, 690], dtype=int64)

This appears to be the best approach so far based on the slight investigation I have done reading the reviews and comparing them to the query. 
Though some results are being overpowered by the word "vegan" allowing ice cream and cafes/bakeries to be on the top 10...